# Lesson 4.4: Practice Prep Churn By Tier

This Practice Exercise produces the first half of Hannah's Friday deliverable: churn rate by plan tier, with supporting numbers.


## Set up
Import all needed libraries and set options for Pandas.
Load CSV file needed 

In [1]:
# Imports
import pandas as pd
import numpy as np

#Set options for Pandas to display all columns and all flot will be rounded by second decimal 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load CSV file
df = pd.read_csv(
    'finflow-customers-200k.csv',
    # Ensure all dates are set as date types
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

#Print to check if file loaded correctly
print(f"Loaded {len(df):,} rows from finflow-customers-200k.csv")
print(f"Shape: {df.shape}")

Loaded 200,000 rows from finflow-customers-200k.csv
Shape: (200000, 16)


## Question 1: Per-tier customer counts.

How many customers does FinFlow have in each plan tier? Use value_counts() or a groupby with .size(). Print the result.

In [2]:
# Plan tier counts
print("Plan tier distribution:")
print(df['plan_tier'].value_counts())

Plan tier distribution:
plan_tier
Basic         119881
Pro            60229
Enterprise     19890
Name: count, dtype: int64


Basic is about 59.9% of the total (Expected: ~60%) 

Pro is about 30.1% of the total (Expected: ~30%)

Enterprise is about 9.9% of the total (Expected: ~10%)

## Question 2: Per-tier churn rate.

What is the churn rate (proportion of customers where active == False) in each plan tier? Express as a percentage. Use the groupby + boolean-mean pattern from Step 11 of the Guided Example.

In [3]:
# Churn rate each plan tier as a percentage
churn_rate_by_tier = (1 - df.groupby('plan_tier')['active'].mean()) * 100
print("Churn rate by plan tier:")
print(f"{churn_rate_by_tier}")

Churn rate by plan tier:
plan_tier
Basic        19.93
Enterprise   19.88
Pro          20.09
Name: active, dtype: float64


All churn rates are expected to be 20%. All tiers are within 0.12 percentage point 

## Question 3: Per-tier monthly recurring revenue from active customers.

What is the sum of monthly_fee for active customers in each plan tier? Hint: filter to active customers first, then groupby.



In [4]:
# Find total monthly_fee of active customers in each plan tier
active_customers = df[df['active']]
active_mrr_per_tier = active_customers.groupby('plan_tier')['monthly_fee'].sum()
print(active_mrr_per_tier)

plan_tier
Basic         958880.16
Enterprise   3171264.00
Pro          1443388.71
Name: monthly_fee, dtype: float64


Plan tier with highest mrr with active customers is Enterprise with $3,171,264.00

Plan tier with second highest mrr with active customer is Pro with $1,443,388.71

Plan tier with lowest mrr with active customer is Basic with $958,880.16

## Question 4: Per-tier average sessions and support tickets, split by active state.

For each combination of (plan_tier, active), compute the mean of total_sessions and the mean of support_tickets_opened. Use a multi-column groupby with .agg(). Unstack the result so the table reads cleanly.

In [5]:
# Multiple metrics per tier in one DataFrame grouped by state
summary_by_state = df.groupby(['state', 'active']).agg({
    'total_sessions': 'mean',
    'support_tickets_opened': 'mean'
}).unstack()

print("Per-state summary:")
print(summary_by_state)

Per-state summary:
       total_sessions        support_tickets_opened      
active          False  True                   False True 
state                                                    
AK              26.51 209.92                   3.81  1.98
AL              25.38 207.35                   4.02  2.01
AR              26.22 209.65                   3.97  2.00
AS              25.02 205.82                   3.96  2.04
AZ              25.02 208.77                   3.96  2.04
CA              25.55 210.60                   4.00  1.98
CO              24.48 207.19                   3.86  2.05
CT              26.84 211.63                   3.85  1.97
DC              25.80 209.72                   4.08  2.00
DE              25.85 211.25                   4.16  1.98
FL              24.53 213.55                   4.07  2.03
FM              25.88 208.44                   3.92  2.03
GA              26.04 211.65                   3.90  2.05
GU              25.56 212.00                   4.00  

## Question 5: A consolidated summary table.

Build a single DataFrame that has one row per plan_tier and the following columns:

customer_count
churn_rate_pct
active_mrr (total monthly_fee from active customers)
avg_sessions_active (mean total_sessions among active customers)
avg_sessions_churned (mean total_sessions among churned customers)
This is the table that goes into Hannah's Friday memo. Make it clean enough to paste into a slide.

Hint for question 5: you can build this in steps. Compute each column as a separate Series indexed by plan_tier, then assemble them into a final DataFrame with pd.DataFrame({column_dict}). Or do it in one chained operation with .agg() and a custom function. Either path is fine; the second is more idiomatic but the first is easier to debug.

In [6]:
# Compute each column as a separate Series indexed by plan_tier

# Creating each series 
customer_count = df.groupby("plan_tier").size()
churn_rate_pct = (1 - df.groupby("plan_tier")["active"].mean()) * 100
active_mrr = df[df["active"]].groupby("plan_tier")["monthly_fee"].sum()
avg_sessions_active = df[df["active"]].groupby("plan_tier")["total_sessions"].mean()
avg_sessions_churned = df[~df["active"]].groupby("plan_tier")["total_sessions"].mean()

# Combine the results into one DataFrame
final_summary = pd.DataFrame({
    "customer_count": customer_count,
    "churn_rate_pct": churn_rate_pct,
    "active_mrr": active_mrr,
    "avg_sessions_active": avg_sessions_active,
    "avg_sessions_churned": avg_sessions_churned,
})

print(final_summary)

            customer_count  churn_rate_pct  active_mrr  avg_sessions_active  \
plan_tier                                                                     
Basic               119881           19.93   958880.16               209.48   
Enterprise           19890           19.88  3171264.00               210.15   
Pro                  60229           20.09  1443388.71               210.41   

            avg_sessions_churned  
plan_tier                         
Basic                      25.51  
Enterprise                 25.58  
Pro                        25.51  


The plan tier with the highest churn rate is Pro, which hit expectations since churn rate for all plan tier is about 20%. This means that lowest and highest churn rate is just probability and nothing else. 

The plan tier with the highest active_mrr is Enterprise with $3,171,264.00 and the one with the largest customer count is Basic with 119,881 customers

On average, sessions for active and churned customers are huge. Active is about 8x bigger than churned for each tiers suggesting on this data only that engagement is strongly linked to retention. 